##### Benefits of Multi-Agent Architecture
1. Context Isolation
2. Specialization
3. Parallelism
4. Fault Isolation

* Rule of thumb: Use multi-agent when a task requires more than 3 distinct knowledge domains or more than 50K tokens of context. Below that threshold, a single well-prompted agent is simpler, cheaper, and often faster.

##### Orchestration Patterns
1. Sequential Pipeline
2. Parallel Fan-Out / Fan-In
3. Hierarchical Delegation. A top-level coordinator delegates to team leads, who further delegate to workers. This creates a tree of management -- mirroring how large engineering teams actually operate.
4. Conditional Routing:A router agent inspects each incoming task and directs it to the most appropriate specialist. Different inputs take different paths through the system.

* The "agent-sized chunk" heuristic: Each subtask should be completable within 20-30 tool calls. If a subtask requires more, it should be decomposed further. If it requires fewer than 5, it's probably too granular and should be merged with another.

###### Task Decomposition Strategies
1. Fixed Decomposition: Predefined stages that always run in the same order, regardless of the input. The pipeline is designed once and executed many times.
2. Dynamic Decomposition: A coordinator agent analyzes the task and creates subtasks on the fly. The number of subtasks, their nature, and their dependencies are determined at runtime.
3. Hybrid: Fixed Phases, Dynamic Sub-tasks: Most production systems use this approach. The high-level phases are fixed (e.g., Plan → Build → Test → Deploy), but within each phase, the system dynamically decides what work is needed.

##### State Management and Communication
###### Shared State Patterns
1. State File on Disk (JSON): The simplest pattern. A single JSON file on disk that all agents read from and write to. The coordinator manages writes to prevent conflicts.
2. Event-Driven (WebSocket / SSE): Agents publish events as they make progress. Other agents (and the UI) subscribe to these events. State is reconstructed from the event stream.
3. Database-Backed: Agents read from and write to a shared database. Transactions ensure consistency. Queries enable complex state lookups.
###### Inter-Agent Communication
1.  Hub-and-Spoke (Through the Coordinator): Agents never communicate directly. All messages flow through the coordinator. Agent A sends results to the coordinator, which decides what to pass to Agent B.
2. Via Shared Files: Agents read from and write to shared file locations. Agent A writes analysis-results.json, Agent B reads it. The file system is the communication channel.
3. Via Database: Agents write their results to database tables. Other agents query those tables for input. This is hub-and-spoke with the database as the hub.

** Structured Inter-Agent Message **
{
  "source_agent": "security-reviewer",
  "timestamp": "2025-01-15T14:30:00Z",
  "confidence": 0.92,
  "findings": [
    {
      "severity": "high",
      "file": "src/auth/login.py",
      "line": 47,
      "issue": "SQL injection via unsanitized user input",
      "recommendation": "Use parameterized query"
    }
  ],
  "summary": "1 high-severity, 2 medium-severity issues found. No critical vulnerabilities."
}

##### The Jumpstarter Pattern
The **Jumpstarter pattern** is a production-proven approach to multi-phase autonomous code generation. It has been used to generate over 50,000 lines of production code in a single run, with minimal human intervention.

{
  "current_phase": "api",
  "completed_phases": ["scaffold", "database"],
  "remaining_phases": ["api", "frontend", "tests", "qa"],
  "iteration": 3,
  "max_iterations": 15,
  "phase_results": {
    "scaffold": { "status": "complete", "files_created": 12 },
    "database": { "status": "complete", "tables": 8 }
  }
}

* Agents: Scaffold Architect, Data Architect, API Architect, Frontend Architect, QA Validator

* The Stop Hook Pattern: The key innovation is the **stop hook**. After each Claude Code session completes (or hits its context limit), the stop hook checks **.jumpstarter-state.json**. If there are remaining phases, it automatically restarts Claude with a fresh context window, pointed at the next phase. This allows the system to generate far more code than fits in a single context window.

##### Failure Handling and Recovery
###### Agent-Level Retries
1. Retry 2-3 times maximum. If an agent fails 3 times on the same input, it is likely a systematic issue, not a transient failure.
2. Vary the approach on retry. Don't just re-run the same prompt. Simplify the task, provide more context, or use a different model.
3. Exponential backoff for rate limits. Wait 1s, then 2s, then 4s. Respect API rate limits.

###### Fallback Agents
If a specialist agent fails, try a generalist. A dedicated security reviewer might fail on an unusual codebase, but a **general-purpose Opus agent** can often produce a reasonable (if less targeted) analysis

# Pseudocode: fallback agent strategy
try:
    result = run_specialist_agent("security-reviewer", task)
except AgentFailure:
    # Specialist failed -- fall back to generalist
    result = run_generalist_agent("opus-general", task)
    result.metadata["fallback"] = True

###### Graceful Degradation
When one agent in a parallel group fails, continue with partial results rather than failing the entire system. Three out of four reviewers completing is better than zero.

###### Checkpoint Recovery
Save state after each successful phase. If the system crashes in phase 4, restart from the phase 4 checkpoint instead of re-running phases 1 through 3. This is exactly what .jumpstarter-state.json enables.

###### The max_iterations Safety Valve
Every multi-agent loop must have a hard upper bound on iterations. Without it, a confused coordinator can loop forever, burning tokens and producing nothing useful. Set max_iterations based on the expected complexity of the task, plus a comfortable margin.

###### Error Propagation
When an agent fails, the error context must flow up to the coordinator in a structured format.

{
  "agent": "security-reviewer",
  "status": "failed",
  "error": "Rate limit exceeded after 3 retries",
  "attempts": 3,
  "partial_results": {
    "files_reviewed": 12,
    "issues_found": 2
  },
  "suggestion": "Retry in 60 seconds or use fallback agent"
}



